# 06 — Final Aggregation & Report

Pull together every metric from notebooks 01–05 into:
- `summary.csv` — master table
- comparison plots
- `report.md` — narrative summary

In [ ]:
%cd /content/repo
import sys
if '/content/repo/scripts' not in sys.path:
    sys.path.insert(0, '/content/repo/scripts')

from utils import RESULTS, load_records
import json, pandas as pd, numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

runs = pd.DataFrame(load_records())
print('runs:', len(runs))
runs.head() if len(runs) else None

In [ ]:
summaries = {}
for name in ['uncond', 'dna', 'sm', 'enzyme']:
    p = RESULTS / f'{name}_summary.json'
    if p.exists():
        summaries[name] = json.loads(p.read_text())
print('Loaded summary files:', list(summaries))

In [ ]:
refold_path = RESULTS / 'refold.json'
refold = pd.DataFrame(json.loads(refold_path.read_text())) if refold_path.exists() else pd.DataFrame()
print('refold rows:', len(refold))

## Master table

In [ ]:
def _s_per_design(metrics):
    if not isinstance(metrics, dict):
        return None
    return metrics.get('s_per_design') or metrics.get('mean_s_per_design')

if len(runs):
    runs['s_per_design'] = runs['metrics'].apply(_s_per_design)
    master = runs[['model', 'task', 'target', 'length', 'n_designs', 's_per_design']].copy()
    if len(refold):
        rf = refold.groupby(['model', 'task']).agg(
            refold_pass_rate=('pass', 'mean'),
            mean_rmsd=('rmsd', 'mean'),
            mean_plddt=('plddt', 'mean'),
        ).reset_index()
        master = master.merge(rf, on=['model', 'task'], how='left')
    master.to_csv(RESULTS / 'summary.csv', index=False)
    print(master)
else:
    master = pd.DataFrame()
    print('No runs to summarise.')

## Radar plot

In [ ]:
def safe_mean(s, default=0):
    if s is None or len(s) == 0: return default
    s = pd.to_numeric(s, errors='coerce').dropna()
    return float(s.mean()) if len(s) else default

if len(master):
    has_refold = 'refold_pass_rate' in master.columns
    has_rmsd = 'mean_rmsd' in master.columns

    metrics_data = {}
    for m in ['rfd3', 'chroma']:
        sub = master.query("model == @m")
        s = safe_mean(sub['s_per_design'], default=1)
        d = safe_mean(sub['refold_pass_rate'], default=0) if has_refold else 0
        r = safe_mean(sub['mean_rmsd'], default=10) if has_rmsd else 10
        metrics_data[m] = {
            'speed_inv':    1.0 / max(s, 1e-3),
            'designability': d,
            'rmsd_inv':     1.0 / max(r, 0.1),
        }

    keys = ['speed_inv', 'designability', 'rmsd_inv']
    labels = ['Speed (1/s)', 'Designability', '1 / RMSD']
    norm = {}
    for k in keys:
        mx = max((metrics_data[m][k] for m in metrics_data), default=1) or 1
        norm[k] = {m: metrics_data[m][k] / mx for m in metrics_data}

    angles = np.linspace(0, 2*np.pi, len(keys), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    for m, color in zip(['rfd3', 'chroma'], ['tab:orange', 'tab:blue']):
        vals = [norm[k][m] for k in keys] + [norm[keys[0]][m]]
        ax.plot(angles, vals, lw=2, label=m.upper(), color=color)
        ax.fill(angles, vals, alpha=0.2, color=color)
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels)
    ax.set_yticklabels([])
    ax.legend(loc='upper right')
    ax.set_title('Normalized comparison (higher = better)')
    plt.tight_layout()
    plt.savefig(RESULTS / 'fig_radar.png', dpi=150)
    plt.show()
else:
    print('Skipping radar — no data.')

## Auto-generated report

In [ ]:
def fmt(x, n=2):
    if x is None: return 'n/a'
    try:
        f = float(x)
        return f'{f:.{n}f}' if not np.isnan(f) else 'n/a'
    except (TypeError, ValueError):
        return 'n/a'

lines = ['# Comparative summary: RFdiffusion3 vs Chroma\n']

if len(master):
    has_refold = 'refold_pass_rate' in master.columns
    has_rmsd = 'mean_rmsd' in master.columns
    for m in ['rfd3', 'chroma']:
        sub = master.query("model == @m")
        if len(sub) == 0: continue
        lines.append(f'## {m.upper()}\n')
        lines.append(f'- runs:               **{len(sub)}**')
        lines.append(f'- mean s/design:      **{fmt(sub["s_per_design"].mean(), 1)}**')
        if has_refold:
            lines.append(f'- mean refold pass:   **{fmt(sub["refold_pass_rate"].mean(), 3)}**')
        if has_rmsd:
            lines.append(f'- mean refold RMSD:   **{fmt(sub["mean_rmsd"].mean(), 2)} Å**')
        lines.append('')

    if len(refold):
        lines.append('## Per-task pass rate\n')
        try:
            pivot = refold.groupby(['task', 'model'])['pass'].mean().unstack().round(3)
            lines.append(pivot.to_markdown())
            lines.append('')
        except Exception as e:
            lines.append(f'(pivot failed: {e})')

    if has_refold:
        rfd3 = master.query("model == 'rfd3'")
        chrm = master.query("model == 'chroma'")
        rp = safe_mean(rfd3['refold_pass_rate'])
        cp = safe_mean(chrm['refold_pass_rate'])
        rs = safe_mean(rfd3['s_per_design'])
        cs = safe_mean(chrm['s_per_design'])
        if rp or cp:
            lines.append('## Headline\n')
            winner_p = 'RFdiffusion3' if rp >= cp else 'Chroma'
            winner_s = 'Chroma' if cs and (cs <= rs) else 'RFdiffusion3'
            lines.append(f'- **Designability winner:** {winner_p} '
                         f'({fmt(max(rp, cp), 3)} vs {fmt(min(rp, cp), 3)})')
            min_s = min(rs, cs) if (rs and cs) else (rs or cs)
            max_s = max(rs, cs) if (rs and cs) else 0
            lines.append(f'- **Speed winner:**         {winner_s} '
                         f'({fmt(min_s, 1)} s vs {fmt(max_s, 1)} s/design)')
else:
    lines.append('_No data available — run notebooks 01–05 first._')

report = '\n'.join(lines)
(RESULTS / 'report.md').write_text(report)
print(report)

## Output inventory

```
results/
├── summary.csv          ← master table
├── report.md            ← narrative summary
├── refold.json          ← per-design refolding records
├── refold_summary.csv   ← refolding aggregates
├── uncond_summary.json  ← diversity & SS fractions
├── dna_summary.json     ← DNA binder counts
├── sm_summary.json      ← small-molecule pocket metrics
├── enzyme_summary.json  ← motif RMSD pass rates
├── fig_uncond_efficiency.png
├── fig_uncond_diversity.png
├── fig_sm_burial.png
├── fig_enzyme_motif.png
├── fig_refold_pass.png
└── fig_radar.png
```